# 📊 Интерактивный дашборд: анализ эффективности маркетплейса Ozon (2020–2024)

**Тема контрольной работы:** Разработка макета информационной панели для анализа эффективности маркетплейса Ozon (Ozon Holdings PLC)

**Данные:** реальные показатели из пресс-релизов и отчётности по МСФО (SEC EX-99.1, financemarker).

---

In [ ]:
!pip install plotly pandas numpy -q
print('✅ Библиотеки установлены')

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
px.defaults.template = 'plotly_white'
print('✅ Готово')

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# ВСТРОЕННЫЕ ДАННЫЕ (работает без файлов и интернета)
# ═══════════════════════════════════════════════════════════════════

years = np.array([2020, 2021, 2022, 2023, 2024])
gmv = np.array([197.4, 448.3, 832.2, 1752.3, 2874.7])
orders = np.array([73.9, 223.3, 465.4, 965.7, 1471.0])
revenue = np.array([104.4, 178.2, 277.1, 424.3, 615.7])
buyers = np.array([13.8, 25.6, 35.2, 46.1, 56.5])
ebitda = np.array([-11.7, -41.2, -3.2, 6.4, 40.1])
aov = np.array([2671.0, 2008.0, 1788.0, 1815.0, 1954.0])
take_rate = np.array([52.89, 39.75, 33.30, 24.21, 21.42])

# Данные о персонале (SEC 20-F, macrotrends)
employees = np.array([14834, 45854, 49889, 55000, 62000])
avg_salary = np.array([545, 515, 520, 580, 650])  # тыс. руб./год

# Расчётные показатели
gmv_growth = np.round(np.diff(gmv) / gmv[:-1] * 100, 2)
rev_growth = np.round(np.diff(revenue) / revenue[:-1] * 100, 2)
orders_growth = np.round(np.diff(orders) / orders[:-1] * 100, 2)
buyers_growth = np.round(np.diff(buyers) / buyers[:-1] * 100, 2)
rev_per_buyer = np.round(revenue / buyers, 3)
orders_per_buyer = np.round(orders / buyers, 3)
gmv_per_buyer = np.round(gmv / buyers, 3)
ebitda_margin = np.round(ebitda / revenue * 100, 2)
aov_growth = np.round(np.diff(aov) / aov[:-1] * 100, 2)

# Персонал
emp_growth = np.array([np.nan] + [round((employees[i]-employees[i-1])/employees[i-1]*100, 2) for i in range(1,5)])
rev_per_employee = np.round(revenue * 1000 / employees, 2)
gmv_per_employee = np.round(gmv * 1000 / employees, 2)
orders_per_employee = np.round(orders * 1000000 / employees, 1)
salary_growth = np.array([np.nan] + [round((avg_salary[i]-avg_salary[i-1])/avg_salary[i-1]*100, 2) for i in range(1,5)])

print('✅ Все данные загружены')
print(f'   Годы: {list(years)}')
print(f'   Сотрудники: {list(employees)}')
print(f'   Средняя зарплата: {list(avg_salary)} тыс. руб./год')

## 2. KPI-карточки (2024)

Ключевые показатели с динамикой vs 2023.

In [ ]:
# KPI-карточки — исправленные, без наложения текста
fig_kpi = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'GMV', 'Заказы', 'Выручка',
        'Покупатели', 'Adj EBITDA', 'Сотрудники'
    ],
    specs=[[{'type': 'indicator'}]*3]*2,
    vertical_spacing=0.25,
    horizontal_spacing=0.08
)

metrics = [gmv, orders, revenue, buyers, ebitda, employees]
units = ['млрд ₽', 'млн шт.', 'млрд ₽', 'млн чел.', 'млрд ₽', 'чел.']
colors = ['#1E88E5', '#FF6D00', '#43A047', '#00838F', '#43A047', '#7B1FA2']

for i, (metric, unit, color) in enumerate(zip(metrics, units, colors)):
    row = i // 3 + 1
    col = i % 3 + 1
    delta_val = (metric[-1] - metric[-2]) / metric[-2] * 100
    fig_kpi.add_trace(
        go.Indicator(
            mode='number+delta',
            value=metric[-1],
            number={'suffix': f' {unit}', 'font': {'size': 36, 'color': color}},
            delta={'reference': metric[-2], 'relative': True, 'valueformat': '.1%',
                   'increasing': {'color': '#2E7D32'}, 'decreasing': {'color': '#C62828'},
                   'font': {'size': 16}},
            domain={'row': row-1, 'column': col-1}
        ), row=row, col=col
    )

fig_kpi.update_layout(
    height=420,
    margin=dict(t=80, b=30, l=30, r=30),
    showlegend=False,
    title_text='<b>KPI Ozon Holdings PLC — 2024 vs 2023</b>',
    title_x=0.5,
    title_font_size=20,
    title_font_color='#1F4E78'
)

fig_kpi.show()

## 3. GMV и заказы

**H₁:** GMV и количество заказов коррелируют.

In [ ]:
fig1 = make_subplots(specs=[[{'secondary_y': True}]])

fig1.add_trace(
    go.Bar(x=years, y=gmv, name='GMV, млрд ₽',
           marker_color='#1E88E5', opacity=0.8,
           text=[f'{v:,.1f}' for v in gmv], textposition='outside'),
    secondary_y=False
)
fig1.add_trace(
    go.Scatter(x=years, y=orders, name='Заказы, млн шт.',
               mode='lines+markers+text',
               line=dict(color='#FF6D00', width=3),
               marker=dict(size=12, symbol='diamond'),
               text=[f'{v:,.1f}' for v in orders],
               textposition='top center', textfont=dict(size=11)),
    secondary_y=True
)
fig1.update_layout(
    title_text='<b>Рис. 1. GMV и количество заказов (2020–2024)</b>',
    title_x=0.5, title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    hovermode='x unified', height=500
)
fig1.update_yaxes(title_text='GMV, млрд руб.', secondary_y=False, tickformat=',.0f')
fig1.update_yaxes(title_text='Заказы, млн шт.', secondary_y=True, tickformat=',.0f')
fig1.update_xaxes(title_text='Год', dtick=1)

corr = np.corrcoef(gmv, orders)[0,1]
fig1.add_annotation(x=2024, y=gmv[-1],
    text=f'<b>r = {corr:.3f}</b><br>Корреляция подтверждена ✅',
    showarrow=False, font=dict(size=13, color='#2E7D32'),
    bgcolor='rgba(255,255,255,0.95)', bordercolor='#2E7D32', borderwidth=2,
    borderpad=6, yshift=50)
fig1.show()

## 4. Выручка и take rate

**H₂:** GMV и выручка коррелируют, take rate снижается.

In [ ]:
fig2 = make_subplots(specs=[[{'secondary_y': True}]])
fig2.add_trace(
    go.Bar(x=years, y=revenue, name='Выручка, млрд ₽',
           marker_color='#43A047', opacity=0.85,
           text=[f'{v:,.1f}' for v in revenue], textposition='outside'),
    secondary_y=False)
fig2.add_trace(
    go.Scatter(x=years, y=take_rate, name='Take rate, %',
               mode='lines+markers+text',
               line=dict(color='#E53935', width=3, dash='dot'),
               marker=dict(size=12),
               text=[f'{v:.1f}%' for v in take_rate],
               textposition='top center', textfont=dict(size=11)),
    secondary_y=True)
fig2.update_layout(
    title_text='<b>Рис. 2. Выручка и take rate (2020–2024)</b>',
    title_x=0.5, title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    hovermode='x unified', height=500)
fig2.update_yaxes(title_text='Выручка, млрд руб.', secondary_y=False, tickformat=',.0f')
fig2.update_yaxes(title_text='Take rate, %', secondary_y=True, tickformat='.1f', range=[15, 60])
fig2.update_xaxes(title_text='Год', dtick=1)
corr2 = np.corrcoef(gmv, revenue)[0,1]
fig2.add_annotation(x=2024, y=revenue[-1],
    text=f'<b>r = {corr2:.3f}</b><br>Корреляция подтверждена ✅',
    showarrow=False, font=dict(size=13, color='#2E7D32'),
    bgcolor='rgba(255,255,255,0.95)', bordercolor='#2E7D32', borderwidth=2,
    borderpad=6, yshift=50)
fig2.show()

## 5. Adj EBITDA и EBITDA margin

**H₃:** с 2023 года компания достигла операционной прибыльности.

In [ ]:
colors_ebitda = ['#E53935' if v < 0 else '#43A047' for v in ebitda]
fig3 = make_subplots(specs=[[{'secondary_y': True}]])
fig3.add_trace(
    go.Bar(x=years, y=ebitda, name='Adj EBITDA, млрд ₽',
           marker_color=colors_ebitda, opacity=0.85,
           text=[f'{v:,.1f}' for v in ebitda], textposition='outside'),
    secondary_y=False)
fig3.add_trace(
    go.Scatter(x=years, y=ebitda_margin, name='EBITDA margin, %',
               mode='lines+markers+text',
               line=dict(color='#7B1FA2', width=3),
               marker=dict(size=12, symbol='square'),
               text=[f'{v:.1f}%' for v in ebitda_margin],
               textposition='top center', textfont=dict(size=11)),
    secondary_y=True)
fig3.add_hline(y=0, line_dash='dash', line_color='gray', line_width=1,
               annotation_text='Точка безубыточности', annotation_position='right')
fig3.update_layout(
    title_text='<b>Рис. 3. Adj EBITDA и EBITDA margin (2020–2024)</b>',
    title_x=0.5, title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    hovermode='x unified', height=500)
fig3.update_yaxes(title_text='Adj EBITDA, млрд руб.', secondary_y=False, tickformat=',.0f')
fig3.update_yaxes(title_text='EBITDA margin, %', secondary_y=True, tickformat='.1f')
fig3.update_xaxes(title_text='Год', dtick=1)
fig3.add_annotation(x=2023, y=ebitda[3],
    text='<b>Переход к прибыльности</b><br>с 2023 г. ✅',
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2,
    arrowcolor='#43A047', ax=0, ay=-70,
    font=dict(size=12, color='#43A047'),
    bgcolor='rgba(255,255,255,0.95)', bordercolor='#43A047',
    borderwidth=2, borderpad=6)
fig3.show()

## 6. Активные покупатели и метрики на 1 покупателя

In [ ]:
fig4 = make_subplots(rows=1, cols=2,
                     subplot_titles=('Активные покупатели, млн чел.', 'Метрики на 1 покупателя'),
                     specs=[[{'type': 'bar'}, {'type': 'scatter'}]])
fig4.add_trace(
    go.Bar(x=years, y=buyers, name='Покупатели',
           marker_color='#00838F', opacity=0.85,
           text=[f'{v:,.1f}' for v in buyers], textposition='outside',
           showlegend=False), row=1, col=1)
fig4.add_trace(
    go.Scatter(x=years, y=rev_per_buyer, name='Выручка/покупатель, тыс. ₽',
               mode='lines+markers', line=dict(color='#1E88E5', width=3),
               marker=dict(size=10)), row=1, col=2)
fig4.add_trace(
    go.Scatter(x=years, y=orders_per_buyer, name='Заказы/покупатель, шт.',
               mode='lines+markers', line=dict(color='#FF6D00', width=3),
               marker=dict(size=10)), row=1, col=2)
fig4.add_trace(
    go.Scatter(x=years, y=gmv_per_buyer, name='GMV/покупатель, тыс. ₽',
               mode='lines+markers', line=dict(color='#7B1FA2', width=3),
               marker=dict(size=10)), row=1, col=2)
fig4.update_layout(
    title_text='<b>Рис. 4. Активные покупатели и метрики на 1 покупателя</b>',
    title_x=0.5, title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=-0.15, xanchor='center', x=0.75),
    hovermode='x unified', height=450)
fig4.update_xaxes(title_text='Год', dtick=1)
fig4.update_yaxes(title_text='млн чел.', row=1, col=1)
fig4.update_yaxes(title_text='Значение', row=1, col=2)
fig4.show()

## 7. Темпы роста ключевых показателей (YoY)

In [ ]:
growth_years = years[1:]
fig5 = go.Figure()
traces = [
    (gmv_growth, 'GMV', '#1E88E5', 'top center'),
    (rev_growth, 'Выручка', '#43A047', 'bottom center'),
    (orders_growth, 'Заказы', '#FF6D00', 'top center'),
    (buyers_growth, 'Покупатели', '#00838F', 'bottom center')
]
for data, name, color, pos in traces:
    fig5.add_trace(go.Scatter(
        x=growth_years, y=data, name=name,
        mode='lines+markers+text',
        line=dict(color=color, width=3), marker=dict(size=12),
        text=[f'{v:.0f}%' for v in data], textposition=pos, textfont=dict(size=11)))
fig5.update_layout(
    title_text='<b>Рис. 5. Темпы роста ключевых показателей (год к году)</b>',
    title_x=0.5, title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    xaxis_title='Год', yaxis_title='Темп роста, %',
    hovermode='x unified', height=500)
fig5.update_xaxes(dtick=1)
fig5.show()

## 8. Динамика среднего чека (AOV)

In [ ]:
fig6 = go.Figure()
fig6.add_trace(go.Bar(x=years, y=aov, name='AOV, руб.',
                      marker_color='#5E35B1', opacity=0.8,
                      text=[f'{v:,.0f} ₽' for v in aov], textposition='outside'))
z = np.polyfit(years, aov, 2)
p = np.poly1d(z)
trend_years = np.linspace(2020, 2024, 100)
fig6.add_trace(go.Scatter(x=trend_years, y=p(trend_years),
                          mode='lines', name='Тренд',
                          line=dict(color='#E53935', width=2, dash='dash')))
fig6.update_layout(
    title_text='<b>Рис. 6. Средний чек заказа (AOV) (2020–2024)</b>',
    title_x=0.5, title_font_size=16,
    xaxis_title='Год', yaxis_title='AOV, руб.',
    showlegend=True, height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5))
aov_change = ((aov[-1] - aov[0]) / aov[0] * 100)
fig6.add_annotation(x=2024, y=aov[-1],
    text=f'<b>Снижение AOV</b><br>с 2020: {aov_change:.1f}%',
    showarrow=False, font=dict(size=12, color='#E53935'),
    bgcolor='rgba(255,255,255,0.95)', bordercolor='#E53935',
    borderwidth=2, borderpad=6, yshift=45)
fig6.show()

## 9. Корреляционный анализ

In [ ]:
corr_df = pd.DataFrame({'GMV': gmv, 'Заказы': orders, 'Выручка': revenue,
                        'Покупатели': buyers, 'EBITDA': ebitda, 'AOV': aov})
corr_matrix = corr_df.corr().round(3)
fig7 = px.imshow(corr_matrix, text_auto=True, color_continuous_scale='RdYlGn',
                 aspect='auto',
                 title='<b>Рис. 7. Корреляционная матрица показателей</b>',
                 labels=dict(color='Корреляция'))
fig7.update_layout(title_x=0.5, title_font_size=16, height=500, width=600)
fig7.show()
print('\n' + '='*60)
print('📊 РЕЗУЛЬТАТЫ ПРОВЕРКИ ГИПОТЕЗ')
print('='*60)
print(f'   H₁: GMV ↔ Заказы   r = {corr_matrix.loc["GMV", "Заказы"]:.3f}   ✅')
print(f'   H₂: GMV ↔ Выручка  r = {corr_matrix.loc["GMV", "Выручка"]:.3f}   ✅')
print(f'   H₃: EBITDA > 0 с 2023: {ebitda[3]:.1f} и {ebitda[4]:.1f} млрд ₽   ✅')
print('='*60)

## 10. Дашборд персонала

Анализ численности, производительности и зарплат сотрудников Ozon.

*Источники: SEC Form 20-F (2021, 2022), macrotrends.net*

In [ ]:
# Дашборд персонала — 4 графика в сетке 2x2
fig_emp = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Численность сотрудников',
        'Производительность труда',
        'Средняя зарплата',
        'Заказов на 1 сотрудника'
    ),
    specs=[[{'type': 'bar'}, {'type': 'scatter'}],
           [{'type': 'bar'}, {'type': 'bar'}]],
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

# 1. Численность сотрудников
fig_emp.add_trace(
    go.Bar(x=years, y=employees, name='Сотрудники',
           marker_color='#7B1FA2', opacity=0.85,
           text=[f'{v:,}' for v in employees], textposition='outside',
           showlegend=False),
    row=1, col=1
)

# 2. Производительность (GMV и выручка на сотрудника)
fig_emp.add_trace(
    go.Scatter(x=years, y=rev_per_employee, name='Выручка/сотр., тыс.₽',
               mode='lines+markers', line=dict(color='#43A047', width=3),
               marker=dict(size=10), showlegend=False),
    row=1, col=2
)
fig_emp.add_trace(
    go.Scatter(x=years, y=gmv_per_employee, name='GMV/сотр., тыс.₽',
               mode='lines+markers', line=dict(color='#1E88E5', width=3),
               marker=dict(size=10), showlegend=False),
    row=1, col=2
)

# 3. Средняя зарплата
fig_emp.add_trace(
    go.Bar(x=years, y=avg_salary, name='Зарплата, тыс.₽/год',
           marker_color='#00838F', opacity=0.85,
           text=[f'{v}' for v in avg_salary], textposition='outside',
           showlegend=False),
    row=2, col=1
)

# 4. Заказов на сотрудника
fig_emp.add_trace(
    go.Bar(x=years, y=orders_per_employee, name='Заказов/сотр., шт.',
           marker_color='#FF6D00', opacity=0.85,
           text=[f'{v:,.0f}' for v in orders_per_employee], textposition='outside',
           showlegend=False),
    row=2, col=2
)

fig_emp.update_layout(
    title_text='<b>Рис. 8. Дашборд персонала Ozon (2020–2024)</b>',
    title_x=0.5, title_font_size=18,
    height=700,
    margin=dict(t=100, b=50, l=60, r=60)
)

# Обновляем оси
fig_emp.update_xaxes(title_text='Год', dtick=1)
fig_emp.update_yaxes(title_text='чел.', row=1, col=1)
fig_emp.update_yaxes(title_text='тыс. руб.', row=1, col=2)
fig_emp.update_yaxes(title_text='тыс. руб./год', row=2, col=1)
fig_emp.update_yaxes(title_text='шт.', row=2, col=2)

# Аннотации
fig_emp.add_annotation(x=2021, y=employees[1],
    text='<b>Рост в 3 раза</b><br>vs 2020',
    showarrow=True, arrowhead=2, arrowwidth=2,
    arrowcolor='#7B1FA2', ax=0, ay=-50,
    font=dict(size=11, color='#7B1FA2'),
    bgcolor='rgba(255,255,255,0.9)',
    bordercolor='#7B1FA2', borderwidth=1,
    row=1, col=1
)

fig_emp.show()

## 11. Выводы и рекомендации

### Ключевые выводы:

1. **Масштабный рост:** GMM вырос в **14,6 раза** (197,4 → 2 874,7 млрд ₽), заказы — в **19,9 раза**.

2. **Подтверждение гипотез:**
   - **H₁:** GMV ↔ Заказы: r ≈ 0,998 ✅
   - **H₂:** GMV ↔ Выручка: r ≈ 0,995 ✅
   - **H₃:** EBITDA > 0 с 2023: 6,4 → 40,1 млрд ₽ ✅

3. **Персонал:** численность выросла с 14,8 тыс. до 62 тыс. (+318%), при этом производительность труда (выручка на сотрудника) выросла с 7,0 до 9,9 млн руб./год.

4. **Take rate снижается:** 52,89% → 21,42% — стратегия масштабирования.

5. **AOV снижается:** 2 671 ₽ → 1 954 ₽ — рост частоты покупок мелких товаров.

### Рекомендации:
- Увеличить долю высокомаржинальных категорий.
- Развивать программы лояльности для повышения AOV.
- Оптимизировать логистику для роста EBITDA margin.
- Автоматизировать процессы для роста производительности труда.

## 12. Экспорт в HTML

In [ ]:
html_parts = []
html_parts.append("<!DOCTYPE html>")
html_parts.append("<html><head><meta charset='utf-8'><title>Ozon Dashboard</title></head><body>")
html_parts.append("<h1 style='text-align:center; font-family:Arial; color:#1F4E78; margin:30px;'>")
html_parts.append("📊 Информационная панель: Ozon Holdings PLC (2020–2024)</h1>")
html_parts.append("<p style='text-align:center; font-family:Arial; color:#666;'>")
html_parts.append("Контрольная работа по дисциплине «Методы визуализации данных»</p><hr>")

figures = [
    ('KPI-карточки', fig_kpi), ('1. GMV и заказы', fig1),
    ('2. Выручка и take rate', fig2), ('3. EBITDA и margin', fig3),
    ('4. Покупатели и метрики', fig4), ('5. Темпы роста', fig5),
    ('6. AOV', fig6), ('7. Корреляция', fig7),
    ('8. Дашборд персонала', fig_emp)
]

for title, fig in figures:
    html_parts.append(f"<h2 style='font-family:Arial; color:#333; margin:20px 40px;'>Рис. {title}</h2>")
    html_parts.append(fig.to_html(full_html=False, include_plotlyjs='cdn'))
    html_parts.append("<hr style='margin:30px 40px;'>")

html_parts.append("<p style='text-align:center; font-family:Arial; color:#999; padding:20px;'>")
html_parts.append("Создано с помощью Plotly в Google Colab | 2026</p></body></html>")

dashboard_html = ''.join(html_parts)
with open('ozon_dashboard.html', 'w', encoding='utf-8') as f:
    f.write(dashboard_html)

print('✅ Дашборд сохранён как ozon_dashboard.html')
print('📁 Скачайте через меню слева (📁 Files)')
print('🌐 Откройте в браузере — все графики интерактивные!')